<a href="https://colab.research.google.com/github/el07n/Deep-learning/blob/main/notebooks/01_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SmartPlate AI - Nutrition5k Data Preparation

This notebook downloads the official Nutrition5k overhead RGB subset, applies the project cleaning rules, creates leakage-safe train/validation/test splits, and displays the final dataset statistics.

**Expected final counts:** 2,321 train, 432 validation, and 506 test dishes.


## 1. Download the project source
The notebook uses the exact preprocessing code stored in the GitHub repository.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/el07n/Deep-learning.git"
PROJECT_DIR = Path("/content/Deep-learning")

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(f"Project ready at: {PROJECT_DIR}")

## 2. Set the temporary Colab paths
The raw images and prepared manifest are kept in Colab runtime storage because thousands of small files are slow in Google Drive. They are removed when the runtime is deleted.

In [ ]:
DATASET_ROOT = Path("/content/nutrition5k_dataset")
PREPARED_DIR = Path("/content/smartplate_processed")

print("Raw dataset path:", DATASET_ROOT)
print("Clean output path:", PREPARED_DIR)

## 3. Download the official Nutrition5k subset
This downloads metadata, official split files, and overhead `rgb.png` images only. It does not download the complete 181 GB dataset. The RGB subset is approximately 1.25 GB.

In [ ]:
subprocess.run([
    sys.executable, "-m", "scripts.download_nutrition5k_support_files",
    "--dataset-root", str(DATASET_ROOT),
], check=True)

subprocess.run([
    sys.executable, "-m", "scripts.download_nutrition5k_subset",
    "--dataset-root", str(DATASET_ROOT),
    "--workers", "16",
], check=True)

## 4. Run the cleaning and preprocessing pipeline
The pipeline removes duplicate dish IDs, missing nutrition values, unavailable image paths, invalid nutrition targets, missing/corrupt images, and manually rejected dish IDs. It preserves the official test split and creates a deterministic validation split.

In [ ]:
subprocess.run([
    sys.executable, "-m", "scripts.prepare_nutrition5k",
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(PREPARED_DIR),
    "--max-ingredients", "50",
    "--exclude-ids", "data/exclude_dish_ids.txt",
], check=True)

print("Cleaning completed successfully.")

## 5. Inspect the exact cleaning function

In [ ]:
import inspect
from smartplate.data import clean_manifest

print(inspect.getsource(clean_manifest))

## 6. Display the cleaning report and split statistics

In [ ]:
import json
import pandas as pd
from IPython.display import display

statistics = json.loads((PREPARED_DIR / "dataset_statistics.json").read_text(encoding="utf-8"))
cleaning_table = pd.Series(statistics["cleaning_report"], name="count").to_frame()
split_table = pd.Series(statistics["split_counts"], name="dishes").to_frame()

print("Cleaning report")
display(cleaning_table)
print("Final splits")
display(split_table)
print("Ingredient vocabulary size:", statistics["ingredient_vocabulary_size"])

## 7. Display the generated dataset visualizations

In [ ]:
from IPython.display import Image as DisplayImage

display(DisplayImage(filename=str(PREPARED_DIR / "plots" / "nutrition_distributions.png")))
display(DisplayImage(filename=str(PREPARED_DIR / "plots" / "ingredient_frequencies.png")))

## 8. Verify the prepared files

In [ ]:
for path in sorted(PREPARED_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PREPARED_DIR))

print("\nPrepared manifest:", PREPARED_DIR / "manifest.csv")
print("Use this same Colab runtime for model training, or run the standalone model notebook.")